# Fuente2024

[https://doi.org/10.1016/j.plaphy.2024.109138](https://doi.org/10.1016/j.plaphy.2024.109138)

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.patches import Patch, Rectangle
from mxlpy import Simulator, scan
from mxlpy.parallel import parallelise

from mxlmodels import get_fuente_2024

# This model needs assimulo

## Figure 2

In [ ]:
pfd = np.mean([50, 950])

fig2_m = get_fuente_2024()

fig2_m.update_parameters({
    "f": 1 / 10000,
    "PPFD": pfd,
    "PPFD_add": 950 - pfd
})

s = Simulator(fig2_m)

s.simulate(40000)

fig2_res = s.get_result().unwrap_or_err().get_combined()

In [ ]:

fig2, axs = plt.subplots(3, 2, figsize=(9, 8))

light_diff = fig2_res["osc_light"].diff()
light_color = "#fdb338ff"
dark_color = "#025196ff"

new_res = pd.DataFrame()
new_res["light"] = fig2_res["osc_light"]
new_res["pq_ox"] = fig2_res["PQ"]
new_res["-log_Hlumen"] = -np.log10(fig2_res["H_lumen"] * 10**(-6))
new_res["quencher_active"] = fig2_res["Q_active"]
new_res["atp_stroma_ratio"] = fig2_res["ATP_st"] / get_fuente_2024().get_parameter_values()["AP_tot"]
new_res["ps1_ox"] = fig2_res["PSI_ox"]

for ax, res_str in zip(axs.flatten(), new_res.columns, strict=False):
    points = np.array([new_res.index, new_res[res_str].values]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    colors = np.where(light_diff >= 0, light_color, dark_color)
    lc = LineCollection(segments, colors=colors, linewidths=2)

    ax.add_collection(lc)
    
    ax.set_xlim(new_res.index.min(), new_res.index.max())
    ax.set_ylim(new_res[res_str].min(), new_res[res_str].max())

for ax in axs.flatten():
    ax.set_xlim(2e4, 4e4)
    ax.set_xticks(np.linspace(2e4, 4e4, 5))
    ax.tick_params(axis="x", direction="in")
    
axs[0,0].set_ylabel("Light Intensity (µmol photons m$^{-2}$ s$^{-1}$)")
axs[0,0].set_ylim(0, 1100)
axs[0,0].set_yticks(np.linspace(0, 1000, 6))

axs[0,1].set_ylabel(r"Oxidized PQ (PQ $\cdot$ PSII$^{-1}$)")
axs[0,1].set_ylim(1, 5)
axs[0,1].set_yticks(np.linspace(1, 5, 5))

axs[1,0].set_ylabel(r"Lumen pH")
axs[1,0].set_ylim(5.6, 6.6)
axs[1,0].set_yticks(np.linspace(5.6, 6.6, 6))

axs[1,1].set_ylabel(r"Active quencher (rel. units)")
axs[1,1].set_ylim(0, 1)
axs[1,1].set_yticks(np.linspace(0, 1, 6))

axs[2,0].set_ylabel(r"ATP/Stroma Ratio")
axs[2,0].set_ylim(0, 1)
axs[2,0].set_yticks(np.linspace(0, 1, 6))
axs[2,0].set_xlabel("Time (s)")

axs[2,1].set_ylabel(r"PI$_{ox}$ (rel. units)")
axs[2,1].set_ylim(0, 0.8)
axs[2,1].set_yticks(np.linspace(0, 0.8, 5))
axs[2,1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## Figure 3

In [ ]:
fig3_m = get_fuente_2024()

fig3_m.update_parameters({
    "PPFD_add": 0
})

fig3_res = scan.steady_state(
    model=fig3_m,
    to_scan=pd.DataFrame({
        "PPFD": np.linspace(10, 1000, 100)
    })
).combined

In [ ]:
fig3, axs = plt.subplots(ncols=2, figsize=(12, 4))
    
new_res = pd.DataFrame()
new_res["atp_stroma"] = fig3_res["ATP_st"] / get_fuente_2024().get_parameter_values()["AP_tot"]
new_res["active_quencher"] = fig3_res["Q_active"]
new_res["lumen_pH"] = -np.log10(fig3_res["H_lumen"] * 10**(-6))
new_res["ps1_ox"] = fig3_res["PSI_ox"] / get_fuente_2024().get_parameter_values()["PSI_total"]
new_res["pq_ox"] = fig3_res["PQ"] / get_fuente_2024().get_parameter_values()["PQ_tot"]
new_res["o2"] = fig3_res["O2"]
new_res["npq"] = fig3_res["NPQ"]
new_res["flou"] = fig3_res["Fluo"]

style_dict = {
    "atp_stroma": {
        "color": "#8eb133",
        "label": "ATP"
    },
    "active_quencher": {
        "color": "#e19d20",
        "label": "Active Quencher"
    },
    "lumen_pH": {
        "color": "#000000",
        "label": "Lumen pH"
    },
    "ps1_ox": {
        "color": "#f16431",
        "label": "PI$_{ox}$"
    },
    "pq_ox": {
        "color": "#5f7fb6",
        "label": "Oxidized PQ"
    },
    "o2": {
        "color": "#000000",
        "label": "O$_2$ evolution"
    },
    "npq": {
        "color": "#e29c23",
        "label": "NPQ"
    },
    "flou": {
        "color": "#5b82b5",
        "label": "ChlF"
    }
}

ax_lumen = axs[0].twinx()
ax_o2 = axs[1].twinx()

for col_str in new_res.columns:
    if col_str == "lumen_pH":
        ax = ax_lumen
    elif col_str == "o2":
        ax = ax_o2
    elif col_str in ["npq", "flou"]:
        ax = axs[1]
    else:
        ax = axs[0]
    ax.plot(
        new_res.index,
        new_res[col_str],
        color=style_dict[col_str]["color"],
        linewidth=2
    )
    
for ax in axs:
    ax.set_xlabel(r"Light Intensity (µmol photons $\cdot$ m$^{-2}$ $\cdot$ s$^{-1}$)")
    ax.set_xlim(-10, 1010)
    ax.set_xticks(np.linspace(0, 1000, 6))
    
    ax.add_patch(
        Rectangle(
            (0, ax.get_ylim()[0]),
            50,
            ax.get_ylim()[1] - ax.get_ylim()[0],
            facecolor="#f3f2f0",
            edgecolor="black",
            zorder=-1
        )
    )
    
axs[0].set_ylabel("Concentrations Ratios")
axs[0].set_ylim(0, 0.9)
axs[0].set_yticks(np.linspace(0.2, 0.8, 4))
ax_lumen.set_ylabel("Lumen pH")
ax_lumen.set_ylim(5.0, 6.8)
ax_lumen.set_yticks(np.linspace(5.4, 6.6, 4))

axs[1].set_ylabel("NPQ \n ChlF (rel. units)")
axs[1].set_ylim(0, 1.25)
axs[1].set_yticks(np.linspace(00, 1.2, 7))
ax_o2.set_ylabel(r"O$_2$ rate" + "\n"+ r"(mol O$_2$ $\cdot$ mol PSIIRC$^{-1}$ $\cdot$ s$^{-1}$)")
ax_o2.set_ylim(0, 121)
ax_o2.set_yticks(np.linspace(0, 120, 7))

left_handles = []
right_handles = []

for key, vals in style_dict.items():
    patch = Patch(color=vals["color"], label=vals["label"])
    if key in ["npq", "o2", "flou"]:
        right_handles.append(patch)
    else:
        left_handles.append(patch)
        
for ax, handles in zip([axs[0], axs[1]], [left_handles, right_handles], strict=False):
    ax.legend(
        handles=handles,
        handlelength=1,
        handleheight=1,
        frameon=False
    )

plt.tight_layout()
plt.show()

## Figure 4

In [ ]:
def fig4_parralel_func(dic):
    s = Simulator(dic["model"])
    
    s.update_parameter(dic[""f""], dic["f_val"])
    
    s.simulate(dic["sim_end"])
    
    return s.get_result().unwrap_or_err().get_combined()

fig4_res = None
pfds = [(50, 950), (50, 150)]
Ts = [10000, 1000, 100, 1, 0.1, 0.001]
sim_ends = [50000, 5000, 500, 5, 0.5, 0.005]

for pfd_vals in pfds:
    model_copy = copy.deepcopy(get_fuente_2024())
    model_copy.update_parameters({
        "PPFD": np.mean(pfd_vals),
        "PPFD_add": pfd_vals[1] - np.mean(pfd_vals)
    })

    res = parallelise(
        fn=fig4_parralel_func,
        inputs=[(T, {"model": model_copy, "f_val": 1/T, "sim_end": s, ""f"": "f"}) for i, (T, s) in enumerate(zip(Ts, sim_ends))],
    )
    
    res = pd.concat([i[1] for i in res], keys=[i[0] for i in res], names=["T", "time"])
    
    if fig4_res is None:
        fig4_res = pd.concat([res], keys=[f"{pfd_vals}"], names=["pfds"])
    else:
        fig4_res = pd.concat([fig4_res, pd.concat([res], keys=[f"{pfd_vals}"], names=["pfds"])])

In [ ]:
fig4, axs = plt.subplots(ncols=5, nrows=6, figsize=(20, 20))

light_color = "#fdb338ff"
dark_color = "#025196ff"

dash_pattern = (0, (1, 2))

xlims = [(2e4, 4e4), (2e3, 4e3), (2e2, 4e2), (2e0, 4e0), (2e-1, 4e-1), (2e-3, 4e-3)]
ylims = [(1, 6), (5.5, 7.0), (0, 1), (0, 1.0), (0, 1)]
titles = [
    r"Oxidized PQ (PQ $\cdot$ PSII$^{-1}$)",
    r"Lumen pH",
    r"Active quencher (rel. units)",
    r"ATP / Total Adenosines",
    r"PI$_{ox}$ (rel. units)"
]

for pfds in fig4_res.index.get_level_values("pfds").unique():
    new_res_pfds = fig4_res.loc[pfds]
    
    current_ls = "solid" if pfds == "(50, 950)" else dash_pattern
    
    for row, T in enumerate(fig4_res.index.get_level_values('T').unique()):
        new_res = new_res_pfds.loc[T]
        true_res = pd.DataFrame()
        true_res["pq_ox"] = new_res["PQ"]
        true_res["-log_Hlumen"] = -np.log10(new_res["H_lumen"] * 10**(-6))
        true_res["quencher_active"] = new_res["Q_active"]
        true_res["atp_stroma_ratio"] = new_res["ATP_st"] / get_fuente_2024().get_parameter_values()["AP_tot"]
        true_res["ps1_ox"] = new_res["PSI_ox"]
        
        light_diff = new_res["osc_light"].diff().fillna(0)
        for col, res_str in enumerate(true_res.columns):
        
            points = np.array([true_res.index, true_res[res_str].values]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            colors = np.where(light_diff >= 0, light_color, dark_color)
            lc = LineCollection(
                segments, 
                colors=colors, 
                linewidths=2, 
                linestyles=current_ls,
                capstyle="butt",
                zorder=1
            )

            axs[row,col].add_collection(lc)
            if current_ls != "solid":
                axs[row,col].plot(true_res[res_str], color="white", lw=2.5, zorder=2, ls=dash_pattern)
            
            axs[row,col].set_ylim(ylims[col])
            axs[row, col].set_yticks(np.linspace(ylims[col][0], ylims[col][1], 6))
            
            if row == 0:
                    axs[0,col].set_title(titles[col])
            if row == len(res.index.get_level_values('T').unique()) - 1:
                axs[row,col].set_xlabel("Time (s)")

        for ax in axs[row, :]:
            ax.set_xlim(xlims[row])
            ax.set_xticks(np.linspace(xlims[row][0], xlims[row][1], 5))
            
        axs[row,0].set_ylabel(f"T = {T} s")
        
plt.show()

## Figure 5

In [ ]:
fig5, axs = plt.subplots(ncols=3, nrows=6, figsize=(10, 15))

light_color = "#fdb338ff"
dark_color = "#025196ff"

dash_pattern = (0, (1, 2))

xlims = [(2e4, 4e4), (2e3, 4e3), (2e2, 4e2), (2e0, 4e0), (2e-1, 4e-1), (2e-3, 4e-3)]
ylims = [(0, 0.5), (0, 1.5), (0, 120)]
titles = [
    r"ChlF (rel. units)",
    r"NPQ",
    r"O$_2$ rate" + "\n" + r"(mol O$_2$ $\cdot$ mol PSIIRC$^{-1}$ $\cdot$ s$^{-1}$)"
]

for pfds in fig4_res.index.get_level_values("pfds").unique():
    new_res_pfds = fig4_res.loc[pfds]
    
    current_ls = "solid" if pfds == "(50, 950)" else dash_pattern
    
    for row, T in enumerate(fig4_res.index.get_level_values('T').unique()):
        new_res = new_res_pfds.loc[T]
        true_res = pd.DataFrame()
        true_res["flou"] = new_res["Fluo"]
        true_res["npq"] = new_res["NPQ"]
        true_res["o2"] = new_res["O2"]
        
        light_diff = new_res["osc_light"].diff().fillna(0)
        for col, res_str in enumerate(true_res.columns):
        
            points = np.array([true_res.index, true_res[res_str].values]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            colors = np.where(light_diff >= 0, light_color, dark_color)
            lc = LineCollection(
                segments, 
                colors=colors, 
                linewidths=2, 
                linestyles=current_ls,
                capstyle="butt",
                zorder=1
            )

            axs[row,col].add_collection(lc)
            if current_ls != "solid":
                axs[row,col].plot(true_res[res_str], color="white", lw=2.5, zorder=2, ls=dash_pattern)
            
            axs[row,col].set_ylim(ylims[col])
            axs[row, col].set_yticks(np.linspace(ylims[col][0], ylims[col][1], 6))
            
            if row == 0:
                    axs[0,col].set_title(titles[col])
            if row == len(res.index.get_level_values('T').unique()) - 1:
                axs[row,col].set_xlabel("Time (s)")

        for ax in axs[row, :]:
            ax.set_xlim(xlims[row])
            ax.set_xticks(np.linspace(xlims[row][0], xlims[row][1], 5))
            
        axs[row,0].set_ylabel(f"T = {T} s")
            
plt.tight_layout()
plt.show()